<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_1_5_MLR_Ames_Part5_NestedCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 5
## Nested CV

Author: Brad Sheese

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Preprocessing & Pipeline tools
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Settings
%matplotlib inline
pd.set_option('display.max_columns', None)

In [ ]:
warnings.filterwarnings('ignore')

# Download the cleaning module from GitHub
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

data_url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_housing_ames.txt'
X_train, X_test, y_train, y_test = load_and_clean_ames(data_url)
print(f"Cleaned X_train shape: {X_train.shape}")
print(f"Cleaned X_test shape: {X_test.shape}")

# Nested CV uses the full dataset — it replaces the train/test split paradigm
# rather than adding onto it. Reconstructing X and y from the already-cleaned
# splits lets us pass 100% of the data to the outer cross-validation loop.
X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])
print(f"\nFull dataset for Nested CV: {X.shape}")

# Part 5: Advanced Evaluation with Nested Cross-Validation

In Part 4, we tuned Ridge, Lasso, and Elastic Net and then compared all three against the same test set to pick a winner. There is a subtle problem baked into that workflow: every time you consult a test set to make a decision — even the decision of *which model to report* — a small amount of information from that test set leaks into your choices. The more models you compare against the same test set, the more optimistic the winning score becomes.

**Nested Cross-Validation** solves this by eliminating the held-out test set entirely. Two layers of cross-validation keep hyperparameter tuning and performance evaluation fully independent.

---

### What Is the Final Product?

This is the most important thing to understand before writing a line of code.

If you run a 5-fold outer loop and a 5-fold inner loop, the computer trains hundreds of models. The outer loop produces 5 final evaluations, each using a model with potentially *different* hyperparameters. So which of those 5 models do you give to your boss? Which one do you deploy?

**The answer is: NONE OF THEM.**

The final product of Nested CV is **NOT** a deployable model. It is a **trustworthy, unbiased number** — e.g., *"Our process produces a Mean Absolute Error of roughly $13,250 on unseen houses."* Nested CV is strictly an evaluation tool. It proves that your entire methodology — your scaling, your algorithm, and your grid search — works as advertised.

Once Nested CV has proved your pipeline is robust, you step away from it and build the production model:

1.  **Gather Everything:** Take 100% of your Ames data — no test set held back.
2.  **One Last Grid Search:** Run `GridSearchCV` on all the data to find the globally optimal hyperparameters.
3.  **Final Fit:** Train one model on 100% of the data using those parameters (`refit=True` handles this automatically — recall Part 4).
4.  **Deploy:** You cannot evaluate this specific model (you used all your data to train it), but you can tell stakeholders: *"My Nested CV process guarantees this model will perform with an average error of roughly $13,250."*


### 4. The Mechanics of Nested Cross-Validation: The Core Idea

Standard cross-validation suffers from "optimistic bias" because the test set is repeatedly used to make decisions. Every time you check a new `alpha` and say, *"Oh, alpha=10 gave me a better test score than alpha=100,"* a tiny bit of information about the test set leaks into your model design.

Nested CV solves this information leakage by building an impenetrable wall between **Optimization** (finding the best parameters) and **Evaluation** (measuring how well the model generalizes). 

It accomplishes this by using two completely separate layers of cross-validation, nesting one inside the other.

### 5. Visualizing the Nested Loops

Imagine our full Ames dataset partitioned into 5 equal chunks (folds). Here is how the two loops interact to protect the integrity of the test data:

```text
Full Ames Dataset:[ Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 ]

Outer Fold 1:  [       TRAIN (Folds 1-4)       ] | [ TEST: Fold 5 ]
                 ↳ Inner CV runs strictly here 
                 ↳ Finds best alpha (e.g., α=10)
                 ↳ Evaluate on Fold 5 using α=10 → Outer Score 1

Outer Fold 2:[    TRAIN (Folds 1,2,3,5)      ] | [ TEST: Fold 4 ]
                 ↳ Inner CV runs strictly here 
                 ↳ Finds best alpha (e.g., α=100)
                 ↳ Evaluate on Fold 4 using α=100 → Outer Score 2

... process repeats for all 5 outer folds ...

Final Unbiased Estimate = Average of (Score 1, Score 2, Score 3, Score 4, Score 5)
```

### 6. What Actually Happens in Each Outer Fold?

To make this concrete, let's trace through **Outer Fold 1** using the real dimensions of our Ames housing dataset (roughly 2,925 homes). If we use a 5-fold Nested CV, here is the exact sequence of events:

1.  **The Outer Split (The Wall is Built):** The Outer Loop splits the data into roughly 2,340 houses for training and 585 houses for testing (Fold 5). Fold 5 is immediately locked away.
2.  **The Inner Loop (GridSearchCV):** The Inner Loop takes *only* those 2,340 training houses and does its own internal 5-fold CV. It tests a grid of 100 different `alpha` values. After grinding through the math, it decides that `alpha=10` is the mathematical optimum *for this specific batch of 2,340 houses*.
3.  **The Outer Evaluation (The Test):** A brand new Ridge model is built using `alpha=10` and trained on the 2,340 houses. Finally, Fold 5 is unlocked, and the model predicts the prices of those 585 unseen homes. This yields **Outer Score 1**.
4.  **The Reset & Repeat:** The process resets for **Outer Fold 2**. A different set of 585 houses (Fold 4) is locked away. 
    * *Crucial Concept:* Because the Inner Loop is looking at a slightly different set of 2,340 training houses this time, it might decide that `alpha=100` is the best parameter! **The hyperparameter can and will change between outer folds.** This is a feature, not a bug.

### 7. Why This Is the Gold Standard for Unbiased Evaluation

In Nested CV, the test fold in each outer iteration is truly sterile. 

It was never used to train the model's coefficients (the weights of the house features), and—more importantly—**it was never used to influence the data scientist's choices** (the hyperparameters). 

When we average the 5 outer scores at the end, we aren't evaluating a single, static model. We are evaluating the **overall intelligence of our machine learning pipeline**. The final score tells us: *"If I hand this algorithm a random set of training data, allow it to tune itself, and then test it on new data, it will be off by an average of exactly $X."*

### 8. Implementing Nested Cross-Validation in Python

While the theory of Nested CV sounds incredibly complex—involving loops inside of loops and the training of potentially thousands of models—implementing it in `scikit-learn` is remarkably elegant. You do not need to write complex `for` loops from scratch.

The "trick" to Nested CV in scikit-learn relies on one beautiful piece of architecture: **`GridSearchCV` is treated as an estimator.** 

Just like `Ridge()` or a `Pipeline()` can be passed into a cross-validation function, a fully configured `GridSearchCV` object can be passed into an outer cross-validation function. 

Here is how we map the theory to the code:
*   **The Inner Loop:** `GridSearchCV()`
*   **The Outer Loop:** `cross_val_predict()` or `cross_val_score()`

Let's implement this for our Ridge Regression model on the Ames dataset.

#### **Step-by-Step Implementation**

In [ ]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Define the Base Pipeline (Prevents Data Leakage)
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge())
])

# 2. Define the Hyperparameter Grid
# Testing 100 values of alpha from 0.01 to 10,000
param_grid = {
    'regressor__alpha': np.logspace(-2, 4, 100)
}

# 3. Create the INNER LOOP (GridSearchCV)
# This object knows how to take a chunk of data, split it 5 ways, 
# and find the best alpha.
inner_loop_cv = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=param_grid,
    cv=5, # 5-Fold Inner CV
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# 4. Create the OUTER LOOP (cross_val_predict)
# We pass the entire GridSearchCV object as the estimator!
# For each of the 5 outer folds, it will hand the training data to the Inner Loop,
# wait for it to find the best alpha, and then use that tuned model to predict 
# the held-out outer fold.
print("Running Nested Cross-Validation... (This may take a moment)")

nested_log_preds = cross_val_predict(
    estimator=inner_loop_cv, 
    X=X,      # Notice we pass the FULL dataset (X and y), not a manual train/test split!
    y=y, 
    cv=5      # 5-Fold Outer CV
)

# 5. Convert Predictions back to Real Dollars
nested_real_preds = np.exp(nested_log_preds)
real_y_actual = np.exp(y)

# 6. Calculate Final Unbiased Metrics
nested_mae = mean_absolute_error(real_y_actual, nested_real_preds)
nested_rmse = np.sqrt(mean_squared_error(real_y_actual, nested_real_preds))

print("-" * 40)
print(f"Nested CV Real-Dollar MAE:  ${nested_mae:,.2f}")
print(f"Nested CV Real-Dollar RMSE: ${nested_rmse:,.2f}")
print("-" * 40)


### 9. Interpreting the Code

Take a close look at **Step 4**. 

Notice that we did not pass `X_train` and `y_train` into `cross_val_predict`. We passed the **entire `X` and `y` dataset**. Because Nested CV doesn't rely on a single locked-away Test Set, it utilizes 100% of your data to simulate the testing environment. 

When the `cross_val_predict` function finishes running:
*   It has trained $5 \text{ (outer folds)} \times 5 \text{ (inner folds)} \times 100 \text{ (alphas)} = \mathbf{2,500 \text{ different models}}$.
*   It has predicted the price of every single house in Ames exactly once, using a model that had **never seen that house**, and whose **hyperparameters were completely blind to that house**.

The output (e.g., a Nested MAE of ~$13,250) is the most intellectually honest, mathematically rigorous estimate of your model's real-world performance that you can possibly generate.

### 10. Extracting the "Winning" Alphas (Optional but Insightful)
Remember earlier when we said that the Inner Loop might choose a *different* alpha for every single Outer Fold? 

Because we used `cross_val_predict`, those intermediate `GridSearchCV` objects were discarded after they made their predictions. If you want to "peek under the hood" and see exactly which alphas were chosen across the 5 outer folds, you can run a slightly different outer loop using `cross_validate`:

### Interpreting the Nested CV Result vs. Part 4

In Part 4, we compared Ridge, Lasso, and Elastic Net against the same test set and reported the winner's MAE. Compare that number to the `Nested CV Real-Dollar MAE` printed above.

You will likely find the nested CV estimate is **slightly higher** (worse) than the Part 4 test set winner. This gap is the **optimistic bias** — the inflation that came from repeatedly consulting the same test set to choose between models. Every comparison you made in Part 4 (Ridge vs. Lasso vs. Elastic Net) gave that test set a small vote in your final decision.

The nested CV number is the honest one. It answers: *"If I hand this entire pipeline — cleaning, scaling, tuning — a brand-new city's housing data, what error should I expect?"*

In [ ]:
from sklearn.model_selection import cross_validate

# cross_validate allows us to extract the fitted Inner Loop objects
nested_results = cross_validate(
    estimator=inner_loop_cv,
    X=X,
    y=y,
    cv=5,
    return_estimator=True # This saves the 5 tuned GridSearchCV objects!
)

print("Alphas chosen by the Inner Loop across the 5 Outer Folds:")
for i, grid_search_obj in enumerate(nested_results['estimator']):
    best_alpha = grid_search_obj.best_params_['regressor__alpha']
    print(f"Outer Fold {i+1}: Best Alpha = {best_alpha:.2f}")

**What you will see:** You will likely see alphas that are clustered together (e.g., 14.2, 16.5, 12.0, 14.2, 18.1). This is a great sign! It means your model is stable and the optimal penalty strength doesn't wildly swing depending on which houses are in the training set. If the alphas jumped between 0.01 and 10,000, it would be a red flag that your dataset is highly unstable.

### 11. Building and Deploying the Final Production Model

At this point in our workflow, we have accomplished our primary analytical goal. By using Nested Cross-Validation, we can walk into a stakeholder meeting and confidently say: *"Based on rigorous testing, our model will predict house prices with an average error of roughly $13,250."*

But remember the crucial caveat from the beginning of this section: **Nested CV did not give us a model to deploy.** It gave us 5 different models trained on 5 different subsets of data. 

To build the actual Python object that the software engineering team will plug into the real estate app, we must step out of the Nested CV paradigm and use **100% of our data** to make the model as smart as possible.

#### **Step 1: The Final Grid Search (100% of Data)**
We no longer need to hold anything back for testing because we already proved our methodology works. We will run our `GridSearchCV` one last time on the entire `X` and `y` dataset.

In [ ]:
# We use the inner_loop_cv object we created in the last step.
# Notice we are passing the FULL dataset (X and y).
print("Training the Final Production Model on 100% of the data...")
final_grid_search = inner_loop_cv.fit(X, y)

# What was the absolute best alpha when the model was allowed to see every house?
final_alpha = final_grid_search.best_params_['regressor__alpha']
print(f"Final Production Alpha: {final_alpha:.2f}")


#### **Step 2: The "Refit" Magic**
You might be wondering: *"Okay, it found the best alpha. Do I need to manually train a final Ridge model now?"*

No. Recall from Part 4 that we set `refit=True` explicitly in our `GridSearchCV`. That parameter means: as soon as the search identifies the winning hyperparameters, it automatically trains one final, master model on the entire dataset you provided. We can access this master model via the `.best_estimator_` attribute.

#### **Step 3: Extracting Final Business Insights**
Before we hand the model off, let's look at the top 5 features driving house prices in our final, fully optimized production model.


In [ ]:
# 1. Extract the final trained pipeline
final_pipeline = final_grid_search.best_estimator_

# 2. Extract the Ridge model from inside the pipeline
final_ridge_model = final_pipeline.named_steps['regressor']

# 3. Match coefficients to feature names
final_coefs = pd.DataFrame({
    'Feature': X.columns,
    'Beta_Weight': final_ridge_model.coef_
})

# 4. Show the top 5 positive drivers of price
top_5 = final_coefs.sort_values(by='Beta_Weight', ascending=False).head(5)
print("\n--- Top 5 Drivers of Value in the Final Model ---")
print(top_5)

#### **Step 4: Saving the Model for Deployment**
The final step in any Machine Learning project is serialization—saving the model to your hard drive so it can be loaded into a web application, API, or another script. 

We will use Python's built-in `joblib` library to save the entire pipeline. 

*Crucial Concept:* Because we are saving the `Pipeline`, we are saving the **fitted StandardScaler** alongside the Ridge model. When a user inputs a new house tomorrow, the pipeline will automatically apply the exact same standardization rules it learned from the Ames dataset today.

In [ ]:
import joblib

# Define the filename
model_filename = 'ames_production_ridge_model.pkl'

# Save the best estimator (The full Scaler + Ridge Pipeline)
joblib.dump(final_pipeline, model_filename)

print(f"\nSuccess! Production model saved to disk as: {model_filename}")

---

### **Conclusion of the Ames MLR Series**

Congratulations! You have successfully navigated the entire lifecycle of a professional Machine Learning project. Let's recap the "Level Ups" you achieved throughout this series:

1.  **Data Cleaning (Part 1 & 2):** You handled missing values and transformed categorical text into one-hot encoded variables.
2.  **Preventing Leakage (Part 2 & 3):** You implemented `sklearn` Pipelines to ensure data scaling was isolated to training folds and never leaked into validation or test data.
3.  **Real-World Metrics (Part 3):** You converted abstract log-errors back into Actual US Dollars, allowing business stakeholders to understand the ROI of the model.
4.  **Handling Complexity (Part 3):** You applied Regularization (Ridge/Lasso/Elastic Net) to tame a high-dimensional dataset with 225 features, preventing the model from overfitting to noise.
5.  **Hyperparameter Tuning (Part 4):** You eliminated arbitrary "guessing" by using Grid Search to mathematically pinpoint the optimal penalty strength ($\alpha$).
6.  **Unbiased Evaluation (Part 5):** You implemented Nested Cross-Validation to calculate a mathematically rigorous, unbiased estimate of future performance, proving the model is ready for the real world.
